In [1]:
# 필요한 라이브러리들을 임포트합니다.
from sklearn.linear_model import LogisticRegression # 로지스틱 회귀 모델
from sklearn.ensemble import RandomForestClassifier # 랜덤 포레스트 분류기
from sklearn.model_selection import train_test_split, GridSearchCV # 훈련/테스트 데이터 분할 및 그리드 서치를 위한 모듈
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif # 특성 선택을 위한 모듈 (최고 K개 선택, 분산 임계값, ANOVA F-값)
from sklearn.tree import DecisionTreeClassifier # 결정 트리 분류기
from sklearn.metrics import roc_auc_score, fbeta_score, make_scorer # ROC AUC 점수, F-베타 점수, 커스텀 스코어러 생성
from xgboost import XGBClassifier # XGBoost 분류기
import shap # SHAP(SHapley Additive exPlanations) 라이브러리 (모델 예측 설명)
import matplotlib.pyplot as plt # 데이터 시각화를 위한 라이브러리

import pandas as pd # 데이터 조작 및 분석을 위한 라이브러리
import numpy as np # 수치 계산을 위한 라이브러리
import datetime as dt # 날짜 및 시간 처리를 위한 라이브러리
import json # JSON 데이터 처리를 위한 라이브러리

In [2]:
# 경고 메시지 처리를 위한 모듈
import warnings 

# 'use_label_encoder' 경고만 무시합니다.
warnings.filterwarnings("ignore")

#### prepare "data/initial_dataset.p"

In [3]:
# .xlsx -> .p : 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3

# # 파일 경로 지정
# file_path = 'data/1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.xlsx'

# # 엑셀 파일을 DataFrame으로 읽어오기
# # 기본적으로 첫 번째 시트를 읽어옵니다.
# data_row = pd.read_excel(file_path)

# # Define the file path
# output_file_path = 'data/1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.p'

# # Save the DataFrame to a pickle file
# data_row.to_pickle(output_file_path)

In [4]:
# .xlsx -> .p : 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress

# # 파일 경로 지정
# file_path = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.xlsx'

# # 엑셀 파일을 DataFrame으로 읽어오기
# # 기본적으로 첫 번째 시트를 읽어옵니다.
# data_row = pd.read_excel(file_path)

# # Define the file path
# output_file_path = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.p'

# # Save the DataFrame to a pickle file
# data_row.to_pickle(output_file_path)

In [5]:
# read *.p
pickle_file_path_1 = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.p'
data_row_1 = pd.read_pickle(pickle_file_path_1)
pickle_file_path_2 = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.p'
data_row_2 = pd.read_pickle(pickle_file_path_2)

In [6]:
# 필요한 컬럼만 keep

# 파일 경로 지정
file_path = 'data/cols_to_keep.csv'

# CSV 파일을 DataFrame으로 읽어오기
cols_to_keep_df = pd.read_csv(file_path)

cols_to_keep = cols_to_keep_df.iloc[:, 0].tolist()

data_row_1 = data_row_1[cols_to_keep]

In [7]:
# data_row <= data_row1 data_row2

# data_row_2에서 조인할 컬럼만 선택
columns_to_join = ['DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP',
                  #  'wafer_id', 
                #    'SensorOffsetHot-RoomAfterBake', 
                #    'SensorOffsetHot-ColdAfterBake', 
                   'BG pass/fail']

# 선택한 컬럼으로 data_row_2의 부분집합 DataFrame 생성
data_row_2_subset = data_row_2[columns_to_join]

# data_row_1에 data_row_2의 선택된 컬럼들을 조인 키 'DevID'로 병합
data_row = pd.merge(data_row_1, data_row_2_subset, on='DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP', how='left')

# # 결과 DataFrame 확인
# print(merged_df.head())

In [8]:
initial_dataset = data_row.copy() # 원본 데이터셋 복사
# processed_dataset = initial_dataset.copy() # 원본 데이터셋 복사

In [9]:
# prepare for target

initial_dataset['Pass/Fail_pass'] = ((initial_dataset['soft_bin of FT1'] == 1) &
                   (initial_dataset['soft_bin of FT2'] == 1) &
                   (initial_dataset['soft_bin'] == 1)).astype(int)

In [10]:
# prepare for base model

initial_dataset['band gap dpat'] = initial_dataset['BG pass/fail'].apply(lambda x: 'bandGapFail' if x == 'impossible wafer' else 'ok for band gap')

# 컬럼 이름 변경 딕셔너리 생성
new_column_names = {
    'wafer_id': 'WAFER_NO',
    'DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP': 'DevID'
}

# .rename() 메서드를 사용하여 컬럼 이름 변경 (inplace=True로 원본 데이터프레임에 바로 적용)
initial_dataset.rename(columns=new_column_names, inplace=True)

# 변경된 컬럼 이름 확인
# print(initial_dataset.columns)

In [11]:
# initial_dataset

In [12]:
# 제거할 컬럼 리스트 정의
columns_to_drop = [
    'soft_bin of FT1',
    'soft_bin of FT2',
    'soft_bin',
    'BG pass/fail'
]

# 컬럼 drop (원본 DataFrame을 변경하려면 inplace=True 사용)
# 또는 새로운 DataFrame을 만들려면 processed_dataset = processed_dataset.drop(...) 사용
initial_dataset.drop(columns=columns_to_drop, inplace=True)

In [13]:
# Rename the columns
initial_dataset.rename(columns={'x_pos': 'X', 'y_pos': 'Y'}, inplace=True)

In [14]:
# Radius 컬럼 계산
# np.sqrt() 함수는 각 요소의 제곱근을 계산합니다.
initial_dataset['Radius'] = np.sqrt(initial_dataset['X']**2 + initial_dataset['Y']**2)

In [15]:
initial_dataset.to_pickle("data/initial_dataset.p")

#### scnarios

In [16]:
# import vars and function

from algos.algos import *
from config.config import *

In [17]:
##### preprocess_dataset
# def preprocess_dataset(initial_dataset: pd.DataFrame):
    # return processed_dataset # 전처리된 데이터셋 반환

preprocessed_dataset = preprocess_dataset(initial_dataset)

##### create_train_and_test_data
# def create_train_test_data(
#     preprocessed_dataset: pd.DataFrame,
#     split_parameter: dict = None
# ):
#     return train_data, test_data, split_parameter_info

split_parameter = split_parameter_default
train_data, test_data, split_parameter_info = create_train_test_data(preprocessed_dataset, split_parameter)




     데이터셋 전처리 중...
     전처리 완료!



##############################################################################################################################
# 3) Create Train/Test Split (훈련/테스트 데이터 분할) 
##############################################################################################################################

    훈련 및 테스트 데이터셋 생성 중...
    - 분할 전 필터링 미적용.
    - Feature Generation 미적용.

    - 분할 전 훈련 데이터 클래스 분포: {0.0: 3546, 1.0: 71}
    - 샘플링 미적용


In [15]:
import copy

In [16]:
# ## 과정

# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.metrics import confusion_matrix, accuracy_score
# from sklearn.feature_selection import SelectKBest, f_classif
# from sklearn.pipeline import Pipeline
# from xgboost import XGBClassifier
# import matplotlib.pyplot as plt
# import seaborn as sns

# # 2000개의 피처와 1000개의 샘플을 가진 데이터 생성
# n_samples = 1000
# n_features = 2000
# X = np.random.rand(n_samples, n_features)

# # 타겟 변수 생성 (이진 분류)
# # 일부 피처가 타겟과 상관관계가 있도록 설정
# y = (X[:, 0] * 2 + X[:, 1] * 3 + np.random.randn(n_samples) > 2.5).astype(int)

# # 데이터프레임으로 변환
# X_df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(n_features)])
# y_series = pd.Series(y, name='target')

# # 데이터 분할
# X_train, X_test, y_train, y_test = train_test_split(X_df, y_series, test_size=0.2, random_state=42)
# print(f"학습 데이터 크기: {X_train.shape}")
# print(f"테스트 데이터 크기: {X_test.shape}")


# # 파이프라인 정의
# pipeline = Pipeline([
#     ('feature_selection', SelectKBest(score_func=f_classif)),
#     ('model', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
# ])

# # 탐색할 파라미터 그리드 정의
# # SelectKBest의 k (선택할 피처 수)와 XGBoost의 하이퍼파라미터를 동시에 탐색
# param_grid = {
#     'feature_selection__k': [5, 10, 50, 100],  # 5, 10, 50, 100개의 피처를 선택하며 탐색
#     'model__n_estimators': [50, 100, 200],  # XGBoost의 트리 개수
#     'model__max_depth': [3, 5, 7]  # XGBoost의 최대 깊이
# }

# # GridSearchCV 객체 생성 및 학습
# # n_jobs=-1로 설정하여 병렬 처리 활성화
# grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
# grid_search.fit(X_train, y_train)

# # 최적의 파라미터 조합과 성능 확인
# print("최적의 하이퍼파라미터 조합:")
# print(grid_search.best_params_)
# print(f"\n최고 교차 검증 점수: {grid_search.best_score_:.4f}")


# # 최적의 모델 추출
# best_model = grid_search.best_estimator_

# # 테스트 데이터로 예측
# y_pred = best_model.predict(X_test)

# # 정확도(Accuracy) 계산
# accuracy = accuracy_score(y_test, y_pred)
# print(f"\n테스트 데이터 정확도: {accuracy:.4f}")

# # 혼돈 행렬 생성 및 시각화
# conf_matrix = confusion_matrix(y_test, y_pred)

# # plt.figure(figsize=(8, 6))
# # sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
# #             xticklabels=['Predicted 0', 'Predicted 1'],
# #             yticklabels=['Actual 0', 'Actual 1'])
# # plt.title('Confusion Matrix')
# # plt.ylabel('Actual Label')
# # plt.xlabel('Predicted Label')
# # plt.show()


In [17]:
# ## 과정

# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.metrics import fbeta_score, make_scorer
# from sklearn.feature_selection import SelectKBest, f_classif
# from sklearn.pipeline import Pipeline
# from xgboost import XGBClassifier
# import matplotlib.pyplot as plt
# import seaborn as sns

# # 1. 더미(가상) 데이터 생성 (기존 코드와 동일)
# n_samples = 1000
# n_features = 2000
# X = np.random.rand(n_samples, n_features)
# y = (X[:, 0] * 2 + X[:, 1] * 3 + np.random.randn(n_samples) > 2.5).astype(int)
# X_df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(n_features)])
# y_series = pd.Series(y, name='target')
# X_train, X_test, y_train, y_test = train_test_split(X_df, y_series, test_size=0.2, random_state=42)

# # F2-스코어 커스텀 스코어러 생성
# # Scikit-learn의 'f2'는 'fbeta_score'의 2.0으로 가중치를 준 것이므로, 'f2'로 지정 가능합니다.
# f2_scorer = make_scorer(fbeta_score, beta=2.0)

# # 2. 파이프라인 구성 및 GridSearchCV 실행
# pipeline = Pipeline([
#     ('feature_selection', SelectKBest(score_func=f_classif)),
#     ('model', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
# ])

# param_grid = {
#     'feature_selection__k': [5, 10, 50, 100],
#     'model__n_estimators': [50, 100],
#     'model__max_depth': [3, 5]
# }

# # ⭐️ scoring='f2'로 변경 (또는 f2_scorer)
# grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring=f2_scorer, n_jobs=-1)
# grid_search.fit(X_train, y_train)

# best_model = grid_search.best_estimator_

# # 3. 최적 모델의 피처 중요도 및 선택 여부 저장

# # 최적 모델의 feature_selection 단계 추출
# best_selector = best_model.named_steps['feature_selection']
# # 최적 모델의 model 단계 (XGBoost) 추출
# best_xgb_model = best_model.named_steps['model']

# # SelectKBest에 의해 선택된 피처의 인덱스 확인
# selected_features_indices = best_selector.get_support(indices=True)
# selected_features_names = X_train.columns[selected_features_indices]

# # 모든 피처에 대한 정보 저장용 DataFrame 생성
# feature_info_df = pd.DataFrame({
#     'feature_name': X_train.columns
# })

# # 피처 선택 여부 (True/False) 컬럼 추가
# feature_info_df['is_selected'] = feature_info_df['feature_name'].isin(selected_features_names)

# # 선택된 피처에 대한 중요도 점수를 저장할 딕셔너리 생성 (초기값 0)
# feature_importances = {name: 0 for name in X_train.columns}

# # XGBoost 모델의 피처 중요도 점수 업데이트
# # XGBoost 모델은 선택된 피처에 대해서만 중요도를 계산함
# for i, importance in zip(selected_features_indices, best_xgb_model.feature_importances_):
#     feature_importances[X_train.columns[i]] = importance

# # DataFrame에 피처 중요도 점수 컬럼 추가
# feature_info_df['importance_score'] = feature_info_df['feature_name'].map(feature_importances)

# # 결과를 중요도 순으로 정렬하여 출력
# feature_info_df = feature_info_df.sort_values(by='importance_score', ascending=False)

# print("\n--- 베스트 모델의 피처 중요도 및 선택 여부 ---")
# print(feature_info_df.head(10))  # 상위 10개 피처 출력
# print("\n--- 선택되지 않은 피처 중 일부 ---")
# print(feature_info_df[feature_info_df['is_selected'] == False].head(5))

# # 필요에 따라 CSV 파일로 저장
# feature_info_df.to_csv('feature_importance_and_selection.csv', index=False)

In [18]:
# ## 과정

# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.metrics import fbeta_score, make_scorer
# from sklearn.feature_selection import SelectKBest, f_classif
# from sklearn.pipeline import Pipeline
# from xgboost import XGBClassifier
# from sklearn.metrics import confusion_matrix
# import uuid

# # 1. 더미(가상) 데이터 생성 (기존 코드와 동일)
# n_samples = 1000
# n_features = 2000
# X = np.random.rand(n_samples, n_features)
# y = (X[:, 0] * 2 + X[:, 1] * 3 + np.random.randn(n_samples) > 2.5).astype(int)
# X_df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(n_features)])
# y_series = pd.Series(y, name='target')
# X_train, X_test, y_train, y_test = train_test_split(X_df, y_series, test_size=0.2, random_state=42)

# # F2-스코어 커스텀 스코어러 생성
# f2_scorer = make_scorer(fbeta_score, beta=2.0)

# # 2. 파이프라인 구성 및 GridSearchCV 실행 (기존 코드와 동일)
# pipeline = Pipeline([
#     ('feature_selection', SelectKBest(score_func=f_classif)),
#     ('model', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
# ])

# param_grid = {
#     'feature_selection__k': [5, 10, 50, 100],
#     'model__n_estimators': [50, 100],
#     'model__max_depth': [3, 5]
# }

# grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring=f2_scorer, n_jobs=-1)
# grid_search.fit(X_train, y_train)

# best_model = grid_search.best_estimator_

# # 3. 최적 모델의 피처 중요도 및 선택 여부 저장 (기존 코드와 동일)
# best_selector = best_model.named_steps['feature_selection']
# best_xgb_model = best_model.named_steps['model']
# selected_features_indices = best_selector.get_support(indices=True)
# selected_features_names = X_train.columns[selected_features_indices]

# feature_info_df = pd.DataFrame({
#     'feature_name': X_train.columns
# })
# feature_info_df['is_selected'] = feature_info_df['feature_name'].isin(selected_features_names)
# feature_importances = {name: 0 for name in X_train.columns}

# for i, importance in zip(selected_features_indices, best_xgb_model.feature_importances_):
#     feature_importances[X_train.columns[i]] = importance
# feature_info_df['importance_score'] = feature_info_df['feature_name'].map(feature_importances)
# feature_info_df = feature_info_df.sort_values(by='importance_score', ascending=False)

# print("\n--- 베스트 모델의 피처 중요도 및 선택 여부 ---")
# print(feature_info_df.head(10))
# print("\n--- 선택되지 않은 피처 중 일부 ---")
# print(feature_info_df[feature_info_df['is_selected'] == False].head(5))
# feature_info_df.to_csv('feature_importance_and_selection.csv', index=False)


# ### **성능 요약 DataFrame 생성 및 저장**

# # 4. 테스트 데이터로 성능 평가 및 혼동 행렬 계산
# y_pred = best_model.predict(X_test)
# tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# # 5. ftpn_df.csv와 유사한 형태의 DataFrame 생성
# # 이 데이터프레임은 여러 번의 실험 결과를 기록하고 비교하기 좋습니다.
# performance_summary_df = pd.DataFrame([{
#     'fn': fn,
#     'fp': fp,
#     'tn': tn,
#     'tp': tp,
#     'feature_selector_name': best_model.named_steps['feature_selection'].__class__.__name__,
#     'initial_feature_count': X_train.shape[1],
#     'final_feature_count': len(selected_features_names),
#     'f2_score': fbeta_score(y_test, y_pred, beta=2.0)
# }])

# # 결과 출력 및 저장
# print("\n--- 모델 성능 요약 (ftpn_df.csv 유사) ---")
# print(performance_summary_df)

# # CSV 파일로 저장
# performance_summary_df.to_csv('model_performance_summary.csv', index=False)

In [19]:
# ## 과정

# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.metrics import fbeta_score, make_scorer, confusion_matrix
# from sklearn.pipeline import Pipeline
# from xgboost import XGBClassifier
# import uuid

# # 1. 더미(가상) 데이터 생성 (기존 코드와 동일)
# n_samples = 1000
# n_features = 2000
# X = np.random.rand(n_samples, n_features)
# y = (X[:, 0] * 2 + X[:, 1] * 3 + np.random.randn(n_samples) > 2.5).astype(int)
# X_df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(n_features)])
# y_series = pd.Series(y, name='target')
# X_train, X_test, y_train, y_test = train_test_split(X_df, y_series, test_size=0.2, random_state=42)

# # F2-스코어 커스텀 스코어러 생성
# f2_scorer = make_scorer(fbeta_score, beta=2.0)

# # 2. features_values_dfs.csv 파일 로드 (가상 생성)
# # ⭐️ 가상 데이터의 feature_name에 맞춰 feature_importance_df 생성
# feature_importance_df = pd.DataFrame({
#     'feature_name': X_df.columns,
#     'FeatureFilter_variance': np.random.rand(n_features),
#     'FeatureFilter_target_linear_correlation': np.random.rand(n_features),
#     'FeatureFilter_target_xicor_correlation': np.random.rand(n_features),
#     'SFM_importances': np.random.rand(n_features)
# })

# def run_optimization_for_feature_importance(train_data, target_data, feature_importance_df, importance_column, k_values):
#     """
#     특정 중요도 컬럼을 기준으로 피처를 선택하고 최적의 모델을 찾는 함수
#     """
    
#     # 중요도 컬럼의 값에 따라 상위 K개의 피처를 선택
#     sorted_features = feature_importance_df.sort_values(
#         by=importance_column, ascending=False
#     )['feature_name']

#     best_k = k_values[0]
#     best_score = -1.0
#     best_pipeline = None
#     selected_feature_list = []

#     for k in k_values:
#         top_k_features = sorted_features.head(k).tolist()
        
#         # 최적 피처로만 구성된 데이터셋 준비
#         X_train_filtered = train_data[top_k_features]
        
#         # 모델 훈련 파이프라인 (SelectKBest 대신 피처 직접 선택)
#         pipeline = Pipeline([
#             ('model', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
#         ])
        
#         param_grid = {
#             'model__n_estimators': [50, 100],
#             'model__max_depth': [3, 5]
#         }
        
#         grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring=f2_scorer, n_jobs=-1)
#         grid_search.fit(X_train_filtered, target_data)

#         # 현재 K의 성능 평가
#         if grid_search.best_score_ > best_score:
#             best_score = grid_search.best_score_
#             best_k = k
#             best_pipeline = grid_search.best_estimator_
#             selected_feature_list = top_k_features

#     # 최적 모델의 피처 중요도 및 성능 정보 생성
#     best_xgb_model = best_pipeline.named_steps['model']
    
#     feature_info = pd.DataFrame({
#         'feature_name': train_data.columns
#     })
#     feature_info['is_selected'] = feature_info['feature_name'].isin(selected_feature_list)
#     feature_info['importance_column'] = importance_column
#     feature_importances = {name: 0 for name in train_data.columns}
    
#     # 선택된 피처에 대해서만 중요도 점수를 부여
#     for i, importance in enumerate(best_xgb_model.feature_importances_):
#         if i < len(selected_feature_list):
#             feature_importances[selected_feature_list[i]] = importance
        
#     feature_info['importance_score'] = feature_info['feature_name'].map(feature_importances)
#     feature_info['feature_value_by_importance_column'] = feature_info['feature_name'].map(
#         feature_importance_df.set_index('feature_name')[importance_column]
#     )

#     # 테스트 데이터로 최종 성능 평가
#     y_pred = best_pipeline.predict(X_test[selected_feature_list])
#     tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

#     performance_summary = pd.DataFrame([{
#         'fn': fn,
#         'fp': fp,
#         'tn': tn,
#         'tp': tp,
#         'feature_selector_name': importance_column,
#         'initial_feature_count': X_train.shape[1],
#         'final_feature_count': len(selected_feature_list),
#         'f2_score': fbeta_score(y_test, y_pred, beta=2.0),
#         'importance_column': importance_column
#     }])

#     return feature_info, performance_summary

# # 3. 중요도 컬럼별로 최적화 반복 수행 및 결과 누적
# importance_cols = [
#     'FeatureFilter_variance',
#     'FeatureFilter_target_linear_correlation',
#     'FeatureFilter_target_xicor_correlation',
#     'SFM_importances'
# ]
# k_values = [5, 10, 50, 100]

# all_feature_infos = []
# all_performance_summaries = []

# for col in importance_cols:
#     print(f"\n--- {col} 컬럼 기준 최적화 수행 ---")
#     feat_info, perf_summary = run_optimization_for_feature_importance(
#         X_train, y_train, feature_importance_df, col, k_values
#     )
#     all_feature_infos.append(feat_info)
#     all_performance_summaries.append(perf_summary)

# final_feature_info_df = pd.concat(all_feature_infos, ignore_index=True)
# final_performance_summary_df = pd.concat(all_performance_summaries, ignore_index=True)

# # 4. 결과 출력 및 CSV 저장
# print("\n--- 최종 누적된 피처 중요도 정보 ---")
# print(final_feature_info_df.head(10))
# final_feature_info_df.to_csv('final_feature_info.csv', index=False)

# print("\n--- 최종 누적된 모델 성능 요약 ---")
# print(final_performance_summary_df)
# final_performance_summary_df.to_csv('final_performance_summary.csv', index=False)

In [20]:
# ## 과정

# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.metrics import fbeta_score, make_scorer, confusion_matrix
# from sklearn.pipeline import Pipeline
# from xgboost import XGBClassifier
# import uuid

# # 1. 더미(가상) 데이터 생성 (기존 코드와 동일)
# n_samples = 1000
# n_features = 2000
# X = np.random.rand(n_samples, n_features)
# y = (X[:, 0] * 2 + X[:, 1] * 3 + np.random.randn(n_samples) > 2.5).astype(int)
# X_df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(n_features)])
# y_series = pd.Series(y, name='target')
# X_train, X_test, y_train, y_test = train_test_split(X_df, y_series, test_size=0.2, random_state=42)

# # 2. features_values_dfs.csv 파일 로드 (가상 생성)
# feature_importance_df = pd.DataFrame({
#     'feature_name': X_df.columns,
#     'FeatureFilter_variance': np.random.rand(n_features),
#     'FeatureFilter_target_linear_correlation': np.random.rand(n_features),
#     'FeatureFilter_target_xicor_correlation': np.random.rand(n_features),
#     'SFM_importances': np.random.rand(n_features)
# })

# def run_optimization_for_feature_importance(train_data, target_data, feature_importance_df, importance_column, k_values):
#     """
#     특정 중요도 컬럼을 기준으로 피처를 선택하고 최적의 모델을 찾는 함수

#     Parameters:
#     - train_data (pd.DataFrame): 훈련 데이터
#     - target_data (pd.Series): 타겟 데이터
#     - feature_importance_df (pd.DataFrame): 피처 중요도 정보가 담긴 DataFrame
#     - importance_column (str): 중요도 순위를 결정할 컬럼명
#     - k_values (list): 선택할 피처의 개수 후보 리스트

#     Returns:
#     - pd.DataFrame: 최적화된 피처 중요도 정보
#     - pd.DataFrame: 모델 성능 요약 정보
#     """
#     # ⭐️ make_scorer를 함수 내부로 이동
#     f2_scorer = make_scorer(fbeta_score, beta=2.0)
    
#     # 중요도 컬럼의 값에 따라 상위 K개의 피처를 선택
#     sorted_features = feature_importance_df.sort_values(
#         by=importance_column, ascending=False
#     )['feature_name']

#     best_k = k_values[0]
#     best_score = -1.0
#     best_pipeline = None
#     selected_feature_list = []

#     for k in k_values:
#         top_k_features = sorted_features.head(k).tolist()
        
#         # 최적 피처로만 구성된 데이터셋 준비
#         X_train_filtered = train_data[top_k_features]
        
#         # 모델 훈련 파이프라인 (SelectKBest 대신 피처 직접 선택)
#         pipeline = Pipeline([
#             ('model', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
#         ])
        
#         param_grid = {
#             'model__n_estimators': [50, 100],
#             'model__max_depth': [3, 5]
#         }
        
#         grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring=f2_scorer, n_jobs=-1)
#         grid_search.fit(X_train_filtered, target_data)

#         # 현재 K의 성능 평가
#         if grid_search.best_score_ > best_score:
#             best_score = grid_search.best_score_
#             best_k = k
#             best_pipeline = grid_search.best_estimator_
#             selected_feature_list = top_k_features

#     # 최적 모델의 피처 중요도 및 성능 정보 생성
#     best_xgb_model = best_pipeline.named_steps['model']
    
#     feature_info = pd.DataFrame({
#         'feature_name': train_data.columns
#     })
#     feature_info['is_selected'] = feature_info['feature_name'].isin(selected_feature_list)
#     feature_info['importance_column'] = importance_column
#     feature_importances = {name: 0 for name in train_data.columns}
    
#     # 선택된 피처에 대해서만 중요도 점수를 부여
#     for i, importance in enumerate(best_xgb_model.feature_importances_):
#         if i < len(selected_feature_list):
#             feature_importances[selected_feature_list[i]] = importance
        
#     feature_info['importance_score'] = feature_info['feature_name'].map(feature_importances)
#     feature_info['feature_value_by_importance_column'] = feature_info['feature_name'].map(
#         feature_importance_df.set_index('feature_name')[importance_column]
#     )

#     # 테스트 데이터로 최종 성능 평가
#     y_pred = best_pipeline.predict(X_test[selected_feature_list])
#     tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

#     performance_summary = pd.DataFrame([{
#         'fn': fn,
#         'fp': fp,
#         'tn': tn,
#         'tp': tp,
#         'feature_selector_name': importance_column,
#         'initial_feature_count': X_train.shape[1],
#         'final_feature_count': len(selected_feature_list),
#         'f2_score': fbeta_score(y_test, y_pred, beta=2.0),
#         'importance_column': importance_column
#     }])

#     return feature_info, performance_summary

# # 3. 중요도 컬럼별로 최적화 반복 수행 및 결과 누적
# importance_cols = [
#     'FeatureFilter_variance',
#     'FeatureFilter_target_linear_correlation',
#     'FeatureFilter_target_xicor_correlation',
#     'SFM_importances'
# ]
# k_values = [5, 10, 50, 100]

# all_feature_infos = []
# all_performance_summaries = []

# for col in importance_cols:
#     print(f"\n--- {col} 컬럼 기준 최적화 수행 ---")
#     feat_info, perf_summary = run_optimization_for_feature_importance(
#         X_train, y_train, feature_importance_df, col, k_values
#     )
#     all_feature_infos.append(feat_info)
#     all_performance_summaries.append(perf_summary)

# final_feature_info_df = pd.concat(all_feature_infos, ignore_index=True)
# final_performance_summary_df = pd.concat(all_performance_summaries, ignore_index=True)

# # 4. 결과 출력 및 CSV 저장
# print("\n--- 최종 누적된 피처 중요도 정보 ---")
# print(final_feature_info_df.head(10))
# final_feature_info_df.to_csv('final_feature_info.csv', index=False)

# print("\n--- 최종 누적된 모델 성능 요약 ---")
# print(final_performance_summary_df)
# final_performance_summary_df.to_csv('final_performance_summary.csv', index=False)

In [ ]:
# ### 과정

# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.metrics import fbeta_score, make_scorer, confusion_matrix
# from sklearn.pipeline import Pipeline
# from xgboost import XGBClassifier
# import uuid

# # 1. 더미(가상) 데이터 생성 (기존 코드와 동일)
# n_samples = 1000
# n_features = 2000
# X = np.random.rand(n_samples, n_features)
# y = (X[:, 0] * 2 + X[:, 1] * 3 + np.random.randn(n_samples) > 2.5).astype(int)
# X_df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(n_features)])
# y_series = pd.Series(y, name='target')
# X_train, X_test, y_train, y_test = train_test_split(X_df, y_series, test_size=0.2, random_state=42)

# # 2. features_values_dfs.csv 파일 로드 (가상 생성)
# feature_importance_df = pd.DataFrame({
#     'feature_name': X_df.columns,
#     'FeatureFilter_variance': np.random.rand(n_features),
#     'FeatureFilter_target_linear_correlation': np.random.rand(n_features),
#     'FeatureFilter_target_xicor_correlation': np.random.rand(n_features),
#     'SFM_importances': np.random.rand(n_features)
# })

# def run_optimization_for_feature_importance(train_data, target_data, feature_importance_df, importance_column, k_percentiles):
#     """
#     특정 중요도 컬럼을 기준으로 피처를 선택하고 최적의 모델을 찾는 함수
    
#     Parameters:
#     - train_data (pd.DataFrame): 훈련 데이터
#     - target_data (pd.Series): 타겟 데이터
#     - feature_importance_df (pd.DataFrame): 피처 중요도 정보가 담긴 DataFrame
#     - importance_column (str): 중요도 순위를 결정할 컬럼명
#     - k_percentiles (list): 선택할 피처의 백분위수 후보 리스트 (예: [0.05, 0.1, 0.25, 0.5])

#     Returns:
#     - pd.DataFrame: 최적화된 피처 중요도 정보
#     - pd.DataFrame: 모델 성능 요약 정보
#     """
#     f2_scorer = make_scorer(fbeta_score, beta=2.0)
    
#     # 중요도 컬럼의 값에 따라 상위 K개의 피처를 선택
#     sorted_features = feature_importance_df.sort_values(
#         by=importance_column, ascending=False
#     )['feature_name']
    
#     # 백분위수를 실제 피처 개수로 변환
#     n_features_total = len(sorted_features)
#     k_values = [max(1, int(n_features_total * p)) for p in k_percentiles]
    
#     best_k = k_values[0]
#     best_score = -1.0
#     best_pipeline = None
#     selected_feature_list = []

#     for k in k_values:
#         top_k_features = sorted_features.head(k).tolist()
        
#         # 최적 피처로만 구성된 데이터셋 준비
#         X_train_filtered = train_data[top_k_features]
        
#         # 모델 훈련 파이프라인 (SelectKBest 대신 피처 직접 선택)
#         pipeline = Pipeline([
#             ('model', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
#         ])
        
#         param_grid = {
#             'model__n_estimators': [50, 100],
#             'model__max_depth': [3, 5]
#         }
        
#         grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring=f2_scorer, n_jobs=-1)
#         grid_search.fit(X_train_filtered, target_data)

#         # 현재 K의 성능 평가
#         if grid_search.best_score_ > best_score:
#             best_score = grid_search.best_score_
#             best_k = k
#             best_pipeline = grid_search.best_estimator_
#             selected_feature_list = top_k_features

#     # 최적 모델의 피처 중요도 및 성능 정보 생성
#     best_xgb_model = best_pipeline.named_steps['model']
    
#     feature_info = pd.DataFrame({
#         'feature_name': train_data.columns
#     })
#     feature_info['is_selected'] = feature_info['feature_name'].isin(selected_feature_list)
#     feature_info['importance_column'] = importance_column
#     feature_importances = {name: 0 for name in train_data.columns}
    
#     # 선택된 피처에 대해서만 중요도 점수를 부여
#     for i, importance in enumerate(best_xgb_model.feature_importances_):
#         if i < len(selected_feature_list):
#             feature_importances[selected_feature_list[i]] = importance
        
#     feature_info['importance_score'] = feature_info['feature_name'].map(feature_importances)
#     feature_info['feature_value_by_importance_column'] = feature_info['feature_name'].map(
#         feature_importance_df.set_index('feature_name')[importance_column]
#     )

#     # 테스트 데이터로 최종 성능 평가
#     y_pred = best_pipeline.predict(X_test[selected_feature_list])
#     tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

#     performance_summary = pd.DataFrame([{
#         'fn': fn,
#         'fp': fp,
#         'tn': tn,
#         'tp': tp,
#         'feature_selector_name': importance_column,
#         'initial_feature_count': X_train.shape[1],
#         'final_feature_count': len(selected_feature_list),
#         'f2_score': fbeta_score(y_test, y_pred, beta=2.0),
#         'importance_column': importance_column
#     }])

#     return feature_info, performance_summary

# # 3. 중요도 컬럼별로 최적화 반복 수행 및 결과 누적
# importance_cols = [
#     'FeatureFilter_variance',
#     'FeatureFilter_target_linear_correlation',
#     'FeatureFilter_target_xicor_correlation',
#     'SFM_importances'
# ]
# # ⭐️ 백분위수 후보 리스트로 변경
# k_percentiles = [0.05, 0.1, 0.25, 0.5]

# all_feature_infos = []
# all_performance_summaries = []

# for col in importance_cols:
#     print(f"\n--- {col} 컬럼 기준 최적화 수행 ---")
#     feat_info, perf_summary = run_optimization_for_feature_importance(
#         X_train, y_train, feature_importance_df, col, k_percentiles
#     )
#     all_feature_infos.append(feat_info)
#     all_performance_summaries.append(perf_summary)

# final_feature_info_df = pd.concat(all_feature_infos, ignore_index=True)
# final_performance_summary_df = pd.concat(all_performance_summaries, ignore_index=True)

# # 4. 결과 출력 및 CSV 저장
# print("\n--- 최종 누적된 피처 중요도 정보 ---")
# print(final_feature_info_df.head(10))
# final_feature_info_df.to_csv('final_feature_info.csv', index=False)

# print("\n--- 최종 누적된 모델 성능 요약 ---")
# print(final_performance_summary_df)
# final_performance_summary_df.to_csv('final_performance_summary.csv', index=False)


--- FeatureFilter_variance 컬럼 기준 최적화 수행 ---


c:\Users\ohbok\Taipy\sparc-malfunction-classification-develop\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:31:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\ohbok\Taipy\sparc-malfunction-classification-develop\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:31:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\ohbok\Taipy\sparc-malfunction-classification-develop\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:31:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\ohbok\Taipy\sparc-malfunction-classification-develop\.venv\Lib\site-packages\xgboost\traini


--- FeatureFilter_target_linear_correlation 컬럼 기준 최적화 수행 ---


c:\Users\ohbok\Taipy\sparc-malfunction-classification-develop\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:31:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\ohbok\Taipy\sparc-malfunction-classification-develop\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:31:32] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\ohbok\Taipy\sparc-malfunction-classification-develop\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:31:37] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\ohbok\Taipy\sparc-malfunction-classification-develop\.venv\Lib\site-packages\xgboost\traini


--- FeatureFilter_target_xicor_correlation 컬럼 기준 최적화 수행 ---


c:\Users\ohbok\Taipy\sparc-malfunction-classification-develop\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:31:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\ohbok\Taipy\sparc-malfunction-classification-develop\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:31:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\ohbok\Taipy\sparc-malfunction-classification-develop\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:31:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\ohbok\Taipy\sparc-malfunction-classification-develop\.venv\Lib\site-packages\xgboost\traini


--- SFM_importances 컬럼 기준 최적화 수행 ---


c:\Users\ohbok\Taipy\sparc-malfunction-classification-develop\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:32:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\ohbok\Taipy\sparc-malfunction-classification-develop\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:32:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\ohbok\Taipy\sparc-malfunction-classification-develop\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:32:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\ohbok\Taipy\sparc-malfunction-classification-develop\.venv\Lib\site-packages\xgboost\traini


--- 최종 누적된 피처 중요도 정보 ---
  feature_name  is_selected       importance_column  importance_score  \
0    feature_0        False  FeatureFilter_variance          0.000000   
1    feature_1         True  FeatureFilter_variance          0.011257   
2    feature_2         True  FeatureFilter_variance          0.000000   
3    feature_3         True  FeatureFilter_variance          0.000000   
4    feature_4        False  FeatureFilter_variance          0.000000   
5    feature_5         True  FeatureFilter_variance          0.000205   
6    feature_6        False  FeatureFilter_variance          0.000000   
7    feature_7        False  FeatureFilter_variance          0.000000   
8    feature_8         True  FeatureFilter_variance          0.000000   
9    feature_9        False  FeatureFilter_variance          0.000000   

   feature_value_by_importance_column  
0                            0.030947  
1                            0.581750  
2                            0.672922  
3         

In [ ]:
## 사용

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import fbeta_score, make_scorer, confusion_matrix
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import uuid

# 1. 더미(가상) 데이터 생성 (기존 코드와 동일)
n_samples = 1000
n_features = 2000
X = np.random.rand(n_samples, n_features)
y = (X[:, 0] * 2 + X[:, 1] * 3 + np.random.randn(n_samples) > 2.5).astype(int)
X_df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(n_features)])
y_series = pd.Series(y, name='target')
X_train, X_test, y_train, y_test = train_test_split(X_df, y_series, test_size=0.2, random_state=42)

# 2. features_values_dfs.csv 파일 로드 (가상 생성)
feature_importance_df = pd.DataFrame({
    'feature_name': X_df.columns,
    'FeatureFilter_variance': np.random.rand(n_features),
    'FeatureFilter_target_linear_correlation': np.random.rand(n_features),
    'FeatureFilter_target_xicor_correlation': np.random.rand(n_features),
    'SFM_importances': np.random.rand(n_features)
})

def run_optimization_for_feature_importance(train_data, target_data, feature_importance_df, importance_column, k_percentiles):
    """
    특정 중요도 컬럼을 기준으로 피처를 선택하고 최적의 모델을 찾는 함수
    
    Parameters:
    - train_data (pd.DataFrame): 훈련 데이터
    - target_data (pd.Series): 타겟 데이터
    - feature_importance_df (pd.DataFrame): 피처 중요도 정보가 담긴 DataFrame
    - importance_column (str): 중요도 순위를 결정할 컬럼명
    - k_percentiles (list): 선택할 피처의 백분위수 후보 리스트 (예: [0.05, 0.1, 0.25, 0.5])

    Returns:
    - pd.DataFrame: 최적화된 피처 중요도 정보
    - pd.DataFrame: 모델 성능 요약 정보
    """
    f2_scorer = make_scorer(fbeta_score, beta=2.0)
    
    # 중요도 컬럼의 값에 따라 상위 K개의 피처를 선택
    sorted_features = feature_importance_df.sort_values(
        by=importance_column, ascending=False
    )['feature_name']
    
    # 백분위수를 실제 피처 개수로 변환
    n_features_total = len(sorted_features)
    k_values = [max(1, int(n_features_total * p)) for p in k_percentiles]
    
    best_k = k_values[0]
    best_score = -1.0
    best_pipeline = None
    selected_feature_list = []
    # ⭐️ 최적의 백분위수 값을 저장할 변수
    best_k_percentile = k_percentiles[0]

    for i, k in enumerate(k_values):
        top_k_features = sorted_features.head(k).tolist()
        
        # 최적 피처로만 구성된 데이터셋 준비
        X_train_filtered = train_data[top_k_features]
        
        # 모델 훈련 파이프라인 (SelectKBest 대신 피처 직접 선택)
        pipeline = Pipeline([
            ('model', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
        ])
        
        param_grid = {
            'model__n_estimators': [50, 100],
            'model__max_depth': [3, 5]
        }
        
        grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring=f2_scorer, n_jobs=-1)
        grid_search.fit(X_train_filtered, target_data)

        # 현재 K의 성능 평가
        if grid_search.best_score_ > best_score:
            best_score = grid_search.best_score_
            best_k = k
            best_pipeline = grid_search.best_estimator_
            selected_feature_list = top_k_features
            # ⭐️ 최적의 백분위수 업데이트
            best_k_percentile = k_percentiles[i]

    # 최적 모델의 피처 중요도 및 성능 정보 생성
    best_xgb_model = best_pipeline.named_steps['model']
    
    feature_info = pd.DataFrame({
        'feature_name': train_data.columns
    })
    feature_info['is_selected'] = feature_info['feature_name'].isin(selected_feature_list)
    feature_info['importance_column'] = importance_column
    feature_importances = {name: 0 for name in train_data.columns}
    
    # 선택된 피처에 대해서만 중요도 점수를 부여
    for i, importance in enumerate(best_xgb_model.feature_importances_):
        if i < len(selected_feature_list):
            feature_importances[selected_feature_list[i]] = importance
        
    feature_info['importance_score'] = feature_info['feature_name'].map(feature_importances)
    feature_info['feature_value_by_importance_column'] = feature_info['feature_name'].map(
        feature_importance_df.set_index('feature_name')[importance_column]
    )

    # 테스트 데이터로 최종 성능 평가
    y_pred = best_pipeline.predict(X_test[selected_feature_list])
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    performance_summary = pd.DataFrame([{
        'fn': fn,
        'fp': fp,
        'tn': tn,
        'tp': tp,
        'feature_selector_name': importance_column,
        'initial_feature_count': X_train.shape[1],
        'final_feature_count': len(selected_feature_list),
        'f2_score': fbeta_score(y_test, y_pred, beta=2.0),
        'importance_column': importance_column,
        # ⭐️ 최적 백분위수 컬럼 추가
        'best_k_percentile': best_k_percentile
    }])

    return feature_info, performance_summary

# 3. 중요도 컬럼별로 최적화 반복 수행 및 결과 누적
importance_cols = [
    'FeatureFilter_variance',
    'FeatureFilter_target_linear_correlation',
    'FeatureFilter_target_xicor_correlation',
    'SFM_importances'
]
k_percentiles = [0.05, 0.1, 0.25, 0.5]

all_feature_infos = []
all_performance_summaries = []

for col in importance_cols:
    print(f"\n--- {col} 컬럼 기준 최적화 수행 ---")
    feat_info, perf_summary = run_optimization_for_feature_importance(
        X_train, y_train, feature_importance_df, col, k_percentiles
    )
    all_feature_infos.append(feat_info)
    all_performance_summaries.append(perf_summary)

final_feature_info_df = pd.concat(all_feature_infos, ignore_index=True)
final_performance_summary_df = pd.concat(all_performance_summaries, ignore_index=True)

# 4. 결과 출력 및 CSV 저장
print("\n--- 최종 누적된 피처 중요도 정보 ---")
print(final_feature_info_df.head(10))
final_feature_info_df.to_csv('final_feature_info.csv', index=False)

print("\n--- 최종 누적된 모델 성능 요약 ---")
print(final_performance_summary_df)
final_performance_summary_df.to_csv('final_performance_summary.csv', index=False)